In [1]:
import os
import time

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, from_unixtime, to_date
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    LongType,
    StructField,
    StructType,
)

KAFKA_BOOTSTRAP_SERVERS = "urbangreen-kafka:9092"
KAFKA_TOPIC = "sensor_readings"

MINIO_ENDPOINT = "http://urbangreen-minio:9000"
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER", "minioadmin")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin123")
MINIO_BUCKET = os.getenv("MINIO_STAGING_BUCKET", "staging")

spark = (
    SparkSession.builder
    .appName("sensor-readings-stream-notebook-test")
    .master("spark://urbangreen-spark-master:7077")
    .config("spark.driver.host", "urbangreen-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "7078")
    .config("spark.blockManager.port", "7079")
    .config("spark.executor.memory", "512m")
    .config("spark.executor.cores", "1")
    .config("spark.cores.max", "2")

    # Kafka connector
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.2",
            "org.apache.hadoop:hadoop-aws:3.4.1",
        ])
    )

    # MinIO/S3A config
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("MinIO endpoint:", MINIO_ENDPOINT)
print("Bucket:", MINIO_BUCKET)

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-cf851fcb-4d67-41ff-bd93-fccfa465a68b;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.2 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found com.google.cod

Spark version: 4.0.2
MinIO endpoint: http://urbangreen-minio:9000
Bucket: staging


In [2]:
df = spark.range(1000)
print(df.count())

1000


In [3]:
spark.range(10).selectExpr("CAST(id AS STRING) AS id").show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



In [4]:
SENSOR_SCHEMA = StructType(
    [
        StructField("farm_sensor_id", IntegerType(), nullable=False),
        StructField("farm_id", IntegerType(), nullable=False),
        StructField("sensor_type_id", IntegerType(), nullable=False),
        StructField("value", DoubleType(), nullable=False),
        StructField("timestamp", LongType(), nullable=False),
    ]
)

In [5]:
kafka_batch = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
)

raw_messages = kafka_batch.selectExpr(
    "CAST(key AS STRING) AS key",
    "CAST(value AS STRING) AS value",
    "topic",
    "partition",
    "offset",
    "timestamp AS kafka_timestamp"
)

raw_messages.show(truncate=False)

+---+----------------------------------------------------------------------------------------------------+---------------+---------+------+-----------------------+
|key|value                                                                                               |topic          |partition|offset|kafka_timestamp        |
+---+----------------------------------------------------------------------------------------------------+---------------+---------+------+-----------------------+
|1  |{"farm_sensor_id": 1, "farm_id": 1, "sensor_type_id": 1, "value": 21.103, "timestamp": 1751895569}  |sensor_readings|0        |0     |2026-07-07 13:39:29.588|
|5  |{"farm_sensor_id": 5, "farm_id": 1, "sensor_type_id": 5, "value": 56.98, "timestamp": 1751895569}   |sensor_readings|0        |1     |2026-07-07 13:39:29.588|
|7  |{"farm_sensor_id": 7, "farm_id": 2, "sensor_type_id": 1, "value": 20.802, "timestamp": 1751895569}  |sensor_readings|0        |2     |2026-07-07 13:39:29.588|
|8  |{"farm_sens

In [6]:
parsed_batch = (
    raw_messages
    .select(
        "key",
        "topic",
        "partition",
        "offset",
        "kafka_timestamp",
        from_json(col("value"), SENSOR_SCHEMA).alias("payload")
    )
    .select(
        "key",
        "topic",
        "partition",
        "offset",
        "kafka_timestamp",
        "payload.*"
    )
    .withColumn("event_date", to_date(from_unixtime(col("timestamp"))))
)

parsed_batch.show(20, truncate=False)
parsed_batch.printSchema()

+---+---------------+---------+------+-----------------------+--------------+-------+--------------+-------+----------+----------+
|key|topic          |partition|offset|kafka_timestamp        |farm_sensor_id|farm_id|sensor_type_id|value  |timestamp |event_date|
+---+---------------+---------+------+-----------------------+--------------+-------+--------------+-------+----------+----------+
|1  |sensor_readings|0        |0     |2026-07-07 13:39:29.588|1             |1      |1             |21.103 |1751895569|2025-07-07|
|5  |sensor_readings|0        |1     |2026-07-07 13:39:29.588|5             |1      |5             |56.98  |1751895569|2025-07-07|
|7  |sensor_readings|0        |2     |2026-07-07 13:39:29.588|7             |2      |1             |20.802 |1751895569|2025-07-07|
|8  |sensor_readings|0        |3     |2026-07-07 13:39:29.588|8             |2      |2             |58.261 |1751895569|2025-07-07|
|11 |sensor_readings|0        |4     |2026-07-07 13:39:29.588|11            |2     

In [7]:
from pyspark.sql.functions import count, when

null_check = parsed_batch.select(
    count(when(col("farm_sensor_id").isNull(), True)).alias("null_farm_sensor_id"),
    count(when(col("farm_id").isNull(), True)).alias("null_farm_id"),
    count(when(col("sensor_type_id").isNull(), True)).alias("null_sensor_type_id"),
    count(when(col("value").isNull(), True)).alias("null_value"),
    count(when(col("timestamp").isNull(), True)).alias("null_timestamp"),
    count(when(col("event_date").isNull(), True)).alias("null_event_date"),
)

null_check.show()

+-------------------+------------+-------------------+----------+--------------+---------------+
|null_farm_sensor_id|null_farm_id|null_sensor_type_id|null_value|null_timestamp|null_event_date|
+-------------------+------------+-------------------+----------+--------------+---------------+
|                  0|           0|                  0|         0|             0|              0|
+-------------------+------------+-------------------+----------+--------------+---------------+



In [8]:
import os

print("MINIO_ROOT_USER:", os.getenv("MINIO_ROOT_USER"))
print("MINIO_ROOT_PASSWORD exists:", os.getenv("MINIO_ROOT_PASSWORD") is not None)
print("MINIO_STAGING_BUCKET:", os.getenv("MINIO_STAGING_BUCKET"))

MINIO_ROOT_USER: minioadmin
MINIO_ROOT_PASSWORD exists: True
MINIO_STAGING_BUCKET: staging


In [9]:
test_output_path = f"s3a://{MINIO_BUCKET}/raw/kafka/{KAFKA_TOPIC}_notebook_batch_test/"

(
    parsed_batch
    .select(
        "farm_sensor_id",
        "farm_id",
        "sensor_type_id",
        "value",
        "timestamp",
        "event_date",
    )
    .write
    .mode("overwrite")
    .partitionBy("event_date")
    .parquet(test_output_path)
)

print("Written to:", test_output_path)

26/07/09 11:20:50 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


Written to: s3a://staging/raw/kafka/sensor_readings_notebook_batch_test/


In [10]:
stream_output_path = f"s3a://{MINIO_BUCKET}/raw/kafka/{KAFKA_TOPIC}_notebook_stream_test/"
stream_checkpoint_path = f"s3a://{MINIO_BUCKET}/_checkpoints/kafka/{KAFKA_TOPIC}_notebook_stream_test/"

In [11]:
kafka_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)

parsed_stream = (
    kafka_stream
    .select(
        from_json(col("value").cast("string"), SENSOR_SCHEMA).alias("payload")
    )
    .select("payload.*")
    .withColumn("event_date", to_date(from_unixtime(col("timestamp"))))
)

query = (
    parsed_stream
    .writeStream
    .format("parquet")
    .option("path", stream_output_path)
    .option("checkpointLocation", stream_checkpoint_path)
    .outputMode("append")
    .partitionBy("event_date")
    .trigger(processingTime="10 seconds")
    .start()
)

print("Streaming started")
print("Output:", stream_output_path)
print("Checkpoint:", stream_checkpoint_path)

Streaming started
Output: s3a://staging/raw/kafka/sensor_readings_notebook_stream_test/
Checkpoint: s3a://staging/_checkpoints/kafka/sensor_readings_notebook_stream_test/


26/07/09 11:31:05 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/07/09 11:31:18 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000} milliseconds, but spent 12586 milliseconds


In [12]:
query.status

{'message': 'Waiting for next trigger',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [13]:
stream_result = spark.read.parquet(stream_output_path)

stream_result.show(20, truncate=False)
stream_result.groupBy("event_date").count().show(truncate=False)
stream_result.printSchema()

+--------------+-------+--------------+-------+----------+----------+
|farm_sensor_id|farm_id|sensor_type_id|value  |timestamp |event_date|
+--------------+-------+--------------+-------+----------+----------+
|1             |1      |1             |22.479 |1783582132|2026-07-09|
|5             |1      |5             |65.781 |1783582132|2026-07-09|
|7             |2      |1             |22.553 |1783582132|2026-07-09|
|8             |2      |2             |63.548 |1783582132|2026-07-09|
|11            |2      |5             |65.667 |1783582132|2026-07-09|
|15            |3      |3             |574.043|1783582132|2026-07-09|
|17            |3      |5             |66.221 |1783582132|2026-07-09|
|21            |4      |3             |587.039|1783582132|2026-07-09|
|22            |4      |4             |6.242  |1783582132|2026-07-09|
|23            |4      |5             |66.911 |1783582132|2026-07-09|
|25            |5      |1             |22.517 |1783582132|2026-07-09|
|27            |5   

+----------+-----+
|event_date|count|
+----------+-----+
|2025-11-17|1800 |
|2026-07-08|6750 |
|2026-04-30|1800 |
|2026-07-09|10350|
|2026-02-11|1800 |
|2025-10-18|1800 |
|2026-01-25|1800 |
|2025-11-21|1800 |
|2026-03-31|1800 |
|2025-08-04|1800 |
|2026-02-10|1800 |
|2026-06-16|1800 |
|2026-06-13|1800 |
|2025-12-27|1800 |
|2026-04-01|1800 |
|2025-11-19|1800 |
|2026-01-27|1800 |
|2025-10-09|1800 |
|2026-06-14|1800 |
|2026-07-07|2700 |
+----------+-----+
only showing top 20 rows
root
 |-- farm_sensor_id: integer (nullable = true)
 |-- farm_id: integer (nullable = true)
 |-- sensor_type_id: integer (nullable = true)
 |-- value: double (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- event_date: date (nullable = true)



26/07/09 12:00:42 WARN StandaloneAppClient$ClientEndpoint: Connection to urbangreen-spark-master:7077 failed; waiting for master to reconnect...
26/07/09 12:00:42 WARN StandaloneSchedulerBackend: Disconnected from Spark cluster! Waiting for reconnection...
26/07/09 12:00:42 WARN StandaloneAppClient$ClientEndpoint: Connection to urbangreen-spark-master:7077 failed; waiting for master to reconnect...
26/07/09 12:00:56 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.5: Remote RPC client disassociated. Likely due to containers exceeding thresholds, or network issues. Check driver logs for WARN messages.
26/07/09 12:00:59 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.4: Remote RPC client disassociated. Likely due to containers exceeding thresholds, or network issues. Check driver logs for WARN messages.
